# Feedback Loop — Medha v0.4.0

This notebook demonstrates the **feedback loop** feature introduced in Medha 0.4.0.
It lets callers mark a cache hit as correct or incorrect, accumulate counters per entry,
and optionally auto-invalidate entries that receive too many incorrect signals.

| Feature | How it works |
|---|---|
| `cache.feedback(question, correct=True)` | Increments `feedback_correct` counter |
| `cache.feedback(question, correct=False)` | Increments `feedback_incorrect` counter |
| `Settings(feedback_incorrect_threshold=N)` | Auto-invalidates when incorrect count reaches N |
| `export_to_dataframe()` | Shows all entries with their feedback counters |

All examples use `InMemoryBackend` — **no Qdrant, no PostgreSQL, no API keys required**.

**Requirements:**
```bash
pip install "medha-archai[fastembed]"
```

## Section 1 — Setup

Initialise the embedder and a `Medha` instance with default settings (no invalidation threshold).
Seed five question-query pairs so subsequent sections have entries to search and rate.
A `mock_llm` dict simulates a Text-to-SQL model — no real LLM calls are made.

In [ ]:
import asyncio
import pandas as pd

from medha import Medha, InMemoryBackend, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter

# Embedder — runs fully locally, no API key needed
embedder = FastEmbedAdapter(
    model_name="BAAI/bge-small-en-v1.5",
    cache_dir="./.fastembed_cache",
)

# Settings with no feedback threshold (manual mode)
settings = Settings(backend_type="memory")

# Mock LLM: simulates a Text-to-SQL model (no API key needed)
mock_llm = {
    "How many users are registered?": "SELECT COUNT(*) FROM users",
    "What is the total revenue this year?": "SELECT SUM(amount) FROM invoices WHERE YEAR(created_at) = YEAR(NOW())",
    "Show the top 10 most expensive products": "SELECT * FROM products ORDER BY price DESC LIMIT 10",
    "Count orders placed today": "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()",
    "What is the average employee salary?": "SELECT AVG(salary) FROM employees",
}

# Seed pairs
_seed_pairs = list(mock_llm.items())

cache = Medha("feedback_demo", embedder=embedder, settings=settings)
await cache.start()

for question, sql in _seed_pairs:
    await cache.store(question, sql)

print(f"Seeded {len(_seed_pairs)} entries into '{cache._collection_name}'")
print(f"Backend: {type(cache._backend).__name__}")

## Section 2 — Manual Feedback Mode

With no `feedback_incorrect_threshold`, calling `feedback()` simply accumulates counters —
entries are **never auto-invalidated**, giving you full control.

We run five searches, mark some results correct and some incorrect, then call
`export_to_dataframe()` to inspect the `feedback_correct` and `feedback_incorrect`
columns for every entry.

In [ ]:
# Search and record feedback
feedback_log = [
    ("How many users are registered?",         True,  "correct — SQL looks right"),
    ("What is the total revenue this year?",   True,  "correct — matches expectation"),
    ("Show the top 10 most expensive products",False, "wrong — price column is 'unit_price'"),
    ("Count orders placed today",              False, "wrong — timezone not applied"),
    ("What is the average employee salary?",   True,  "correct"),
]

print(f"{'Question':<50s}  {'Correct':>7s}  {'Found':>5s}  Note")
print("-" * 95)

for question, correct, note in feedback_log:
    hit = await cache.search(question)
    found = await cache.feedback(question, correct=correct)
    status = "yes" if correct else "no"
    print(f"{question:<50s}  {status:>7s}  {'✓' if found else '✗':>5s}  {note}")

print()

# Inspect counters via export_to_dataframe()
df = await cache.export_to_dataframe()
cols = ["original_question", "feedback_correct", "feedback_incorrect"]
print("Feedback counters across all entries:")
df[cols].set_index("original_question").sort_index()

## Section 3 — Auto-Invalidation Mode

Set `feedback_incorrect_threshold=3` in `Settings`. Once an entry accumulates
**three** incorrect signals, Medha automatically calls `invalidate()` on it —
the entry is deleted from the backend and evicted from the L1 in-process cache.

We restart with a fresh instance, store one entry, send three incorrect feedbacks,
then confirm the entry is gone: the next search returns `NO_MATCH`.

In [ ]:
await cache.close()

# Restart with auto-invalidation threshold = 3
settings_auto = Settings(backend_type="memory", feedback_incorrect_threshold=3)
cache_auto = Medha("feedback_auto", embedder=embedder, settings=settings_auto)
await cache_auto.start()

target_q = "Show the top 10 most expensive products"
target_sql = mock_llm[target_q]

await cache_auto.store(target_q, target_sql)
print(f"Stored: '{target_q}'")

# Confirm it's there before we start
hit = await cache_auto.search(target_q)
print(f"Before feedback  → strategy: {hit.strategy.value}, query: {hit.generated_query!r}")
print()

# Send 3 incorrect feedbacks
for i in range(1, 4):
    found = await cache_auto.feedback(target_q, correct=False)
    print(f"  feedback #{i} (incorrect) → found={found}")

print()

# Confirm the entry has been auto-invalidated
hit_after = await cache_auto.search(target_q)
print(f"After 3× incorrect → strategy: {hit_after.strategy.value}")
assert hit_after.strategy == SearchStrategy.NO_MATCH, "Expected NO_MATCH after auto-invalidation"
print("✓  Entry auto-invalidated as expected.")

## Section 4 — Mixed Session

A realistic scenario: ten searches with mixed correct/incorrect feedback against
a cache seeded with all five entries and `feedback_incorrect_threshold=3`.

- Entries that accumulate three incorrect signals are auto-invalidated.
- Entries with mostly correct signals survive.

We show the final state via `export_to_dataframe()`, keeping only entries
still present in the backend.

In [ ]:
await cache_auto.close()

settings_mixed = Settings(backend_type="memory", feedback_incorrect_threshold=3)
cache_mixed = Medha("feedback_mixed", embedder=embedder, settings=settings_mixed)
await cache_mixed.start()

for question, sql in _seed_pairs:
    await cache_mixed.store(question, sql)

print(f"Seeded {len(_seed_pairs)} entries (threshold=3).")
print()

# 10-step mixed session
session = [
    ("How many users are registered?",         True),
    ("What is the total revenue this year?",   True),
    ("Show the top 10 most expensive products",False),
    ("Count orders placed today",              False),
    ("What is the average employee salary?",   True),
    ("Show the top 10 most expensive products",False),   # 2nd incorrect
    ("Count orders placed today",              False),   # 2nd incorrect
    ("What is the total revenue this year?",   True),
    ("Show the top 10 most expensive products",False),   # 3rd incorrect → invalidated
    ("Count orders placed today",              False),   # 3rd incorrect → invalidated
]

print(f"{'#':<3}  {'Question':<50s}  {'Correct':>7s}  {'feedback() found':>17s}")
print("-" * 85)

for i, (question, correct) in enumerate(session, 1):
    await cache_mixed.search(question)
    found = await cache_mixed.feedback(question, correct=correct)
    label = "yes" if correct else "no"
    print(f"{i:<3}  {question:<50s}  {label:>7s}  {'✓' if found else 'invalidated':>17s}")

print()

# Final state
df_final = await cache_mixed.export_to_dataframe()
print(f"Surviving entries: {len(df_final)} / {len(_seed_pairs)}")
print()
cols = ["original_question", "feedback_correct", "feedback_incorrect"]
df_final[cols].set_index("original_question").sort_index()

## Section 5 — Production Notes

Guidance for using the feedback loop in production deployments.

### Race-condition caveat

The five non-atomic backends — **Qdrant, Chroma, Weaviate, Azure AI Search, and LanceDB** —
increment feedback counters with a read-modify-write cycle (no compare-and-swap).
Under concurrent `feedback()` calls on the same entry, a write can be lost and the counter
may be slightly off.  
This is the same trade-off already accepted for `update_usage_count`, and for a
human-in-the-loop signal the practical impact is negligible:
in the worst case the auto-invalidation threshold fires **one call late** — the entry
will still be invalidated on the next `feedback()` call.

Backends with **atomic** counter updates: `InMemoryBackend`, `PgVectorBackend`,
`VectorChordBackend`, `ElasticsearchBackend`, `RedisVectorBackend`.

---

### Recommended threshold values

| Threshold | Use case |
|-----------|----------|
| `1` | Zero tolerance — one wrong signal invalidates immediately. Suitable for high-stakes queries (financial, medical). |
| `3` | **Recommended default.** Balances noise resistance against responsiveness. |
| `5` | Conservative — tolerates occasional misclicks or ambiguous feedback. |
| `None` (default) | Manual mode — accumulate counters only; your application decides when to invalidate. |

---

### `feedback_correct` — current behaviour

`feedback_correct` is stored and visible in `export_to_dataframe()`, but it does not
currently act on the cache (e.g. boost confidence scores or extend TTL).
Acting on positive signals — such as increasing the semantic similarity score or
reducing TTL decay — is planned as a future feature.

---

### Using feedback from synchronous code

Medha exposes a sync convenience wrapper for frameworks that are not async-native:

```python
# In a sync context (e.g. Django view, Celery task)
found = cache.feedback_sync("How many users are registered?", correct=False)
```

`feedback_sync()` is a thin wrapper around `asyncio.run()` — do **not** call it from
inside a running event loop (e.g. FastAPI route). Use `await cache.feedback()` there instead.

In [ ]:
# Show current feedback state of the mixed-session cache for reference
df_notes = await cache_mixed.export_to_dataframe()

display_cols = ["original_question", "feedback_correct", "feedback_incorrect"]
print("Final feedback state (surviving entries only):")
print(df_notes[display_cols].to_string(index=False))

await cache_mixed.close()
print("\nAll Medha instances closed.")